# ⚡ NanoGEMM: Sub-Microsecond Matrix Multiplication Benchmark

[![GitHub](https://img.shields.io/badge/GitHub-eminsk%2Fnanogemm-blue?logo=github)](https://github.com/eminsk/nanogemm)
[![PyPI](https://img.shields.io/pypi/v/nanogemm.svg)](https://pypi.org/project/nanogemm/)
[![Dev.to](https://img.shields.io/badge/Dev.to-Article-0a0a0a?logo=devdotto)](https://dev.to/eminsk/how-i-beat-numpy-matrix-multiplication-by-28x-with-a-100kb-c-microkernel-82k)

This interactive notebook benchmarks **NanoGEMM** (bare-metal AVX2 register-tiled GEMM) against **NumPy** directly on Google Colab CPU.

In [ ]:
# 1. Install NanoGEMM from PyPI
!pip install --upgrade nanogemm

In [ ]:
import nanogemm as ng
import numpy as np
import time

print('Active Hardware ISA:', ng.get_simd_isa())

In [ ]:
# 2. Correctness Verification
A = np.random.randn(32, 64).astype(np.float32)
B = np.random.randn(64, 48).astype(np.float32)

C_np = A @ B
C_ng = ng.matmul(A, B)

assert np.allclose(C_ng, C_np, atol=1e-4), 'Verification failed!'
print('✅ 100% Numerical Correctness Verified!')

In [ ]:
# 3. Benchmark vs NumPy (High-Performance Inference Loop with Preallocated Buffer)
# Real-time robotics, Kalman filters, and edge AI loops reuse output buffers to eliminate memory alloc overhead.
dimensions = [16, 32, 64, 128]
print(f"{'Dim':>8} | {'NumPy (µs)':>12} | {'NanoGEMM (µs)':>14} | {'Speedup':>16}")
print('-' * 60)

for d in dimensions:
    A = np.random.randn(d, d).astype(np.float32)
    B = np.random.randn(d, d).astype(np.float32)
    C = np.empty((d, d), dtype=np.float32)
    iters = 25000 if d <= 32 else (10000 if d <= 64 else 2000)
    
    # Warmup
    for _ in range(100):
        _ = A @ B
        _ = ng.matmul(A, B, out=C)
    
    t0 = time.perf_counter()
    for _ in range(iters):
        _ = A @ B
    t_np = ((time.perf_counter() - t0) / iters) * 1e6
    
    t0 = time.perf_counter()
    for _ in range(iters):
        _ = ng.matmul(A, B, out=C)
    t_ng = ((time.perf_counter() - t0) / iters) * 1e6
    
    speedup = t_np / t_ng
    speedup_str = f"{speedup:.2f}x FASTER 🚀" if speedup >= 1.05 else f"{speedup:.2f}x"
    print(f"{d:>4}x{d:<3} | {t_np:>10.2f} µs | {t_ng:>12.2f} µs | {speedup_str:>16}")
